## DL Hackathon Starter Colab
### Date: April 27, 2026
### DSBA+ICEF, HSE University

<small><font color=gray>Notebook authors: <a href="https://www.hse.ru/en/staff/aboldyrev" target="_blank">Alexey Boldyrev</a>, <a href="www.hse.ru/en/staff/mekarpov" target="_blank">Maksim Karpov</a>, <a href="https://www.hse.ru/en/staff/sara" target="_blank">Saraa Ali</a>, <a href="http://wiki.cs.hse.ru/Deep_Learning_DSBA_2025/2026" target="_blank">Stanislav Ryazanov</a>.

[**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

## Problem Description

**Motivation**: One of the most valuable sources of customer information is bank transaction data. In this set, there are many answers to the question: is it possible to predict the gender of a client using information about receipt and payment by bank card? And if so, what is the accuracy of such a prediction?

**Task**: Predict ROC AUC from the probability of gender (0|1) for each `cid` (customer ID) which is missing a gender in the file `gender.csv`.

<small>**CONSENT.** <mark>[ X ]</mark> We consent to sharing our Colab (after the Hackathon ends) with other students/instructors for educational purposes.

In [1]:
from google.colab import userdata

kaggle_token = userdata.get('kaggle_api')

In [2]:
!pip install -q numpy>=2.0.0 kaggle pytorch-lifestream lightgbm pytorch-lightning
!mkdir -p ~/.kaggle                                      # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!echo "$kaggle_token" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token                        # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v hse-dl-hackathon-2026          # hackathon dataset
!kaggle competitions download -c hse-dl-hackathon-2026 >> log                                # download competition dataset as a zip file
!unzip -o *.zip >> log                                              # kaggle dataset is copied as a single file and needs to be unzipped
!kaggle competitions leaderboard -c hse-dl-hackathon-2026 --show                             # print public leaderboard

- competition is now set to: hse-dl-hackathon-2026
Next Page Token = CfDJ8CS0IeAoHcJGgSEc27rBbk7RItG_IJssCvem0m-vsm-qksS8q33Rh8TF7VFaBjDjqPcCncTQQY7XMDbkK8XDoZo
  teamId  teamName        submissionDate              score    
--------  --------------  --------------------------  -------  
15748694  BG              2026-04-27 08:19:46.486000  0.88954  
15748318  Maksim Rodikov  2026-04-27 08:21:31.946000  0.88852  
15748202  N Team          2026-04-27 08:31:41.166000  0.88830  
15748268  AV - AnyaVova   2026-04-27 08:29:11.570000  0.88783  
15748538  F               2026-04-27 08:24:16.190000  0.88750  
15748158  AA              2026-04-27 08:30:00.510000  0.88705  
15748321  AE              2026-04-27 08:24:17.220000  0.88682  
15748186  AC              2026-04-27 08:29:47.186000  0.88676  
15748529  BM              2026-04-27 07:59:04.110000  0.88604  
15748167  G_team          2026-04-27 08:25:55.503000  0.88593  
15748260  AZ              2026-04-27 08:04:54.386000  0.88558  
1574836

In [3]:
import os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print(f'Device: {DEVICE}')

pd.set_option('display.max_rows', 100, 'display.max_columns', 100, 'display.max_colwidth', 100, 'display.precision', 2, 'display.max_rows', 4)

class Timer():
  def __init__(self, lim:'RunTimeLimit'=200): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

# Set all random numbers to ensure that your private LB score for Kaggle is reproducible with IPYNB file, which you submit via LMS.
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

Device: cuda


### Load data

In [4]:
gender = pd.read_csv('gender.csv', index_col='cid')
dfTrx  = pd.read_csv('trx.csv')

tY = gender.dropna()                # labelled CIDs (train)
vY = gender[gender.gender.isna()]   # unlabelled CIDs (test)
print(f'transactions={len(dfTrx):,}  train CIDs={len(tY):,}  test CIDs={len(vY):,}')

transactions=3,749,578  train CIDs=6,400  test CIDs=2,000


1. **trx.csv**, a table with chronological transaction history for ~15 months. Each `cid` can have unequal number of transactions.
    * `cid`, identifier of a bank's client; a non-negative integer
    * `dt`, a date of the transaction; an whole number starting from 0
    * `mcc`, mcc code of the transaction; a whole number; a foreign key for `mcc` in *_mcc.csv* table
    * `ttc`, transaction's type; a whole number; a foreign key for `ttc` in *_ttc.csv* table
    * `amt`, sum of the transactions in some monetary units, rounded to integer. Positive/Negative values are credits/debits to the client's account.
    * `tid`, ID of the register terminal (point of sale or POS) where the transaction was made.
1. **gender.csv**, a table with a numeric `cid` and `gender` (0|1) columns. Predict gender for the rows missing a gender value (i.e. first few thousands). Use the remaining gender values as target values in training your model.
1. **_mcc.csv**, a lookup table with the unique [Merchant Category Codes](https://en.wikipedia.org/wiki/Merchant_category_code) (MCC) and their descriptions in Russian. These might be helpful in locating similar categories using keywords or semantics (for example with sentence vectors).
1. **_ttc.csv**, a lookup table with unique transaction type codes and their verbal descriptions in Russian.

In [5]:
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

⏳ started. You have 200 sec. Good luck!


<hr color=red>

<font size=5>⏳</font> <strong><font color=orange size=5>Your Code, Documentation, Ideas and Timer - All Start Here...</font></strong>

Students: Keep all your definitions, code, documentation **between** ⏳ symbols. Modifying any code outside of the timed playground incurs penalties.

## Preprocessing Pipeline

Explain elements of your preprocessing pipeline i.e. feature engineering, subsampling, clustering, dimensionality reduction, etc.

### 1. Build per-CID features

Two feature blocks: simple `amt` aggregates, and a normalized MCC count vector
(fraction of each CID's transactions falling into each MCC category).
MCC distribution is a strong gender signal: men and women shop at very different merchant types.

In [8]:
import pytorch_lightning as pl
import lightgbm as lgb
from ptls.preprocessing.pandas.pandas_preprocessor import PandasDataPreprocessor
from ptls.nn import TrxEncoder, RnnSeqEncoder
from ptls.frames.coles import CoLESModule
from ptls.frames.coles.split_strategy import SampleSlices
from ptls.data_load.datasets import MemoryMapDataset
from ptls.frames.coles import ColesDataset
from ptls.data_load.utils import collate_feature_dict

# 1. Preprocessing for PyTorch Lifestream
dfTrx['amt_log'] = (np.sign(dfTrx['amt']) * np.log1p(dfTrx['amt'].abs())).astype(np.float32)

preprocessor = PandasDataPreprocessor(
    col_id='cid',
    col_event_time='dt',
    event_time_transformation='none',
    cols_category=['mcc', 'ttc'],
    cols_numerical=['amt_log'],
    return_records=True
)

print("Preprocessing data for ptls...")
dataset = preprocessor.fit_transform(dfTrx)

# Drop very short sequences to ensure stable contrastive learning
dataset = [x for x in dataset if len(x['dt']) >= 10]
print(f"Total valid clients for CoLES: {len(dataset)}")

Creating Dask Server


INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at: inproc://172.28.0.12/5179/1
INFO:distributed.scheduler:  dashboard at:  http://172.28.0.12:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.worker:      Start worker at: inproc://172.28.0.12/5179/4
INFO:distributed.worker:         Listening to:          inproc172.28.0.12
INFO:distributed.worker:          Worker name:                          0
INFO:distributed.worker:         dashboard at:          172.28.0.12:41789
INFO:distributed.worker:Waiting to connect to: inproc://172.28.0.12/5179/1
INFO:distributed.worker:-------------------------------------------------
INFO:distributed.worker:              Threads:                          4
INFO:distributed.worker:               Memory:                  12.67 GiB
INFO:dist

Link Dask Server - http://172.28.0.12:8787/status
Preprocessing data for ptls...
Total valid clients for CoLES: 8349


### 2. Train / validation split and standardization

In [10]:
# 2. Prepare PyTorch Lightning DataLoader for CoLES
coles_ds = ColesDataset(
    MemoryMapDataset(dataset),
    splitter=SampleSlices(split_count=5, cnt_min=10, cnt_max=100)
)
train_dl = DataLoader(coles_ds, batch_size=256, shuffle=True, collate_fn=collate_feature_dict)

# 3. Define ptls TrxEncoder and RnnSeqEncoder
trx_encoder = TrxEncoder(
    embeddings={
        'mcc': {'in': int(dfTrx.mcc.max()) + 2, 'out': 16},
        'ttc': {'in': int(dfTrx.ttc.max()) + 2, 'out': 16}
    },
    numeric_values={'amt_log': 'identity'},
    use_batch_norm=True,
)

# GRU sequence encoder -> maps variable length events to fixed size 256 dim vector
seq_encoder = RnnSeqEncoder(
    trx_encoder=trx_encoder,
    hidden_size=256,
    type='gru'
)

# 4. Wrap into CoLES Module (Contrastive Learning)
model = CoLESModule(
    seq_encoder=seq_encoder,
    optimizer_partial=lambda x: torch.optim.Adam(x, lr=0.002),
)

### 3. Define the model

Multilayer perceptron with two hidden layers with ReLU + dropout. Sigmoid is applied implicitly via `BCEWithLogitsLoss`.

In [11]:
# 5. Train CoLES Model
print("Training CoLES Model (Self-Supervised)...")
trainer = pl.Trainer(
    max_epochs=2, # Using just 2 epochs to fit within 200s constraint!
    accelerator='gpu' if DEVICE == 'cuda' else 'cpu',
    devices=1,
    enable_progress_bar=True
)
trainer.fit(model, train_dl)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Training CoLES Model (Self-Supervised)...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


TypeError: 'NoneType' object is not callable

### 4. Training loop

In [ ]:
# 6. Extract Final Embeddings
print("Extracting sequence embeddings...")
seq_encoder.eval()
seq_encoder.to(DEVICE)

# Save CID ordering to easily map embeddings
cids = [x['cid'] for x in dataset]
inf_dl = DataLoader(
    MemoryMapDataset(dataset),
    batch_size=512,
    shuffle=False,
    collate_fn=collate_feature_dict
)

embs = []
with torch.no_grad():
    for batch in inf_dl:
        out = seq_encoder(batch.to(DEVICE))
        embs.append(out.cpu().numpy())

embs = np.concatenate(embs, axis=0)
df_embeddings = pd.DataFrame(embs, index=cids)
df_embeddings.index.name = 'cid'

print(f"Got Embeddings Shape: {df_embeddings.shape}")

### 5. Predict on test set and save submission

In [ ]:
# 7. Train Supervised Downstream Model (LightGBM)
# Align embeddings with targets
X_train = df_embeddings.reindex(tY.index).fillna(0)
y_train = tY['gender']
X_test  = df_embeddings.reindex(vY.index).fillna(0)

print("Training LightGBM on CoLES embeddings...")
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'verbose': -1
}
dtr = lgb.Dataset(X_train, label=y_train)
lgbm_model = lgb.train(params, dtr, num_boost_round=150)

# Predict
test_p = lgbm_model.predict(X_test)

submission = pd.DataFrame({'cid': vY.index, 'gender': test_p})
submission.to_csv('baseline_submission.csv', index=False, float_format='%.6f')
print(f'Saved baseline_submission.csv  ({len(submission)} rows)')
submission.head()

## **References:**

* Remember to cite your sources here! At the least, your [Dive into Deep Learning](https://d2l.ai/) textbook should be cited. Google Scholar allows you to effortlessly copy/paste an APA citation format for books and publications. Also cite StackOverflow, package documentation, and other meaningful internet resources to help your peers learn from these (and to avoid plagiarism claims).

1. ...
1. ...
1. ...

<font color=green><h4><b>* LLM Documentation if used</b></h4></font>


1. Model and Platform Information  
   - The full name, version and operation mode of the model (e.g., Qwen (Qwen3.6-35B-A3B), Claude Opus 4.7, Gemini 3.1 Pro (Thinking mode)).  
   - A link to the service or platform used (e.g., https://chat.openai.com, https://claude.ai).  
   - If applicable, specify the application or interface (e.g., browser version with built‑in assistant, Telegram bot, IDE extension, etc.).

2. Interaction Record  
   - Provide all prompts and the model’s responses in chronological order (e.g., Prompt 1 – Response 1; Prompt 2 – Response 2, etc.).  
   - Ensure that the full conversation relevant to your submission is preserved and clearly formatted.

3. Reflection  
   - Briefly evaluate the quality and usefulness of the AI’s contribution.  
   - Explain what specific problem or part of the task the LLM helped you address.  
   - State how you verified or modified the output to ensure correctness and originality.

4. Usage Limitations  
   - Using an LLM to generate a complete solution is strictly prohibited.
   - Using LLM to generate documentation on the use of LLM is also prohibited.
   - The tool may be used only for assistance (e.g., drafting, brainstorming, clarifying concepts), not for full problem‑solving or code generation.

5. Academic Integrity  
   - Any submission incorporating LLM‑generated material without the documentation described above will be considered an academic integrity violation and may be treated as plagiarism.  
   - The teaching team reserves the right to determine whether a submission shows signs of unacknowledged AI assistance.


## 💡**Starter Ideas**

1. Richer hand-crafted features (signed-amt, log-magnitude, calendar, category-richness, TTC histogram, MCC×sign cross)
1. 5-fold stratified CV with Out-of-Fold (OOF) predictions
1. LightGBM/CatBoost as a complementary ensemble member
1. OOF-tuned ensemble averaging
1. GRU/Transformer sequence model with embedded (`mcc`, `ttc`, weekday, sign, log|`amt`|-bucket) tokens
1. Self-supervised pre-training and pseudo-labelling using the unlabelled test CIDs
1. Polishing (multi-seed, cosine LR, label smoothing, pos_weight)

<font size=5>⌛</font> <strong><font color=orange size=5>Do not exceed competition's runtime limit!</font></strong>

<hr color=red>

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.